<a href="https://colab.research.google.com/github/fahadabdullah-lab/smap-dispatch-gee-south-texas/blob/main/notebooks/02_data_pull_daily.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Install + imports**

In [ ]:
!pip -q install earthengine-api geemap

import ee
import geemap

### **Cell 2 — Authenticate + initialize**

In [ ]:
ee.Authenticate()
ee.Initialize(project='ee-fafahadabdullah')
print("✅ Earth Engine initialized.")

## **Cell 3 — Parameters (date + AOI + mode)**

In [ ]:
# --- Date (daily-first) ---
TEST_DATE = "2021-08-15"
start = ee.Date(TEST_DATE)
end   = start.advance(1, "day")

# --- Mode ---
USE_TERRA_ONLY = True   # keep True for Notebook 02

# --- AOI (same as Notebook 01) ---
def get_south_texas_aoi():
    bounds = [-97.6, 25.8, -97.0, 26.4]
    return ee.Geometry.Rectangle(bounds)

aoi = get_south_texas_aoi()

print("📅 Date:", TEST_DATE)
print("🧭 AOI loaded")

### **Cell 4 — Map view**

In [ ]:
Map = geemap.Map(center=[26.1, -97.3], zoom=9)
Map.addLayer(aoi, {"color": "yellow"}, "AOI")
Map

### **Cell 5 — SMAP L3 (36 km) pull**

In [ ]:
# ============================================
# SMAP L3 Radiometer (36 km) - SPL3SMP_E
# ============================================
smap_ic = (
    ee.ImageCollection("NASA/SMAP/SPL3SMP_E/005")
    .filterDate(start, end)
    .filterBounds(aoi)
)

print("SMAP images found:", smap_ic.size().getInfo())

# Choose AM soil moisture if available; we’ll adjust if needed
smap_img = ee.Image(smap_ic.first())

# Common band name in this collection is often 'soil_moisture_am'
# We'll list bands to confirm
print("SMAP band names:", smap_img.bandNames().getInfo())

### **Cell 6 — Select soil moisture band + clip**

In [ ]:
# Try common soil moisture band names (AM first)
candidate_bands = ["soil_moisture_am", "soil_moisture_pm", "soil_moisture"]

bands = smap_img.bandNames()

def pick_first_available(cands):
    for b in cands:
        if bands.contains(b).getInfo():
            return b
    return None

sm_band = pick_first_available(candidate_bands)
print("✅ Selected SMAP band:", sm_band)

smap_sm = smap_img.select(sm_band).clip(aoi)

### **Cell 7 — MODIS Terra reflectance (Red + NIR)**

In [ ]:
# ============================================
# MODIS Terra Surface Reflectance (Daily)
# ============================================
mod09_ic = (
    ee.ImageCollection("MODIS/061/MOD09GA")
    .filterDate(start, end)
    .filterBounds(aoi)
)

print("MOD09GA images found:", mod09_ic.size().getInfo())
mod09 = ee.Image(mod09_ic.first())

# Bands: sur_refl_b01 (Red), sur_refl_b02 (NIR)
red = mod09.select("sur_refl_b01")
nir = mod09.select("sur_refl_b02")

print("MOD09GA band names:", mod09.bandNames().getInfo())

### **Cell 8 — Basic MOD09GA QA mask**

In [ ]:
# state_1km bits (common approach):
# bit 0-1: cloud state (0 = clear)
state = mod09.select("state_1km")

cloud_state = state.bitwiseAnd(3)  # extract bits 0-1
clear_mask = cloud_state.eq(0)

red_masked = red.updateMask(clear_mask)
nir_masked = nir.updateMask(clear_mask)

print("✅ Applied basic cloud mask to MOD09GA.")

### **Cell 9 — MODIS Terra LST (MOD11A1) + QC mask**

In [ ]:
# ============================================
# MODIS Terra LST (Daily, 1 km)
# ============================================
lst_ic = (
    ee.ImageCollection("MODIS/061/MOD11A1")
    .filterDate(start, end)
    .filterBounds(aoi)
)

print("MOD11A1 images found:", lst_ic.size().getInfo())
lst_img = ee.Image(lst_ic.first())

# LST Day 1km
lst = lst_img.select("LST_Day_1km").multiply(0.02)  # Kelvin

print("MOD11A1 band names:", lst_img.bandNames().getInfo())

In [ ]:
qc = lst_img.select("QC_Day")
# Keep only pixels with good quality (simplified: bits 0-1 == 0)
lst_good = qc.bitwiseAnd(3).eq(0)
lst_masked = lst.updateMask(lst_good)

print("✅ Applied basic QC mask to MOD11A1.")

### **Cell 10 — Visualize layers**

In [ ]:
# Visualization params
vis_smap = {"min": 0.0, "max": 0.5}
vis_ndvi = {"min": -0.2, "max": 0.9}
vis_lst  = {"min": 290, "max": 330}  # Kelvin

# NDVI
ndvi = nir_masked.subtract(red_masked).divide(nir_masked.add(red_masked)).rename("NDVI")

Map = geemap.Map(center=[26.1, -97.3], zoom=9)
Map.addLayer(aoi, {"color":"yellow"}, "AOI")
Map.addLayer(smap_sm, vis_smap, "SMAP SM (36 km)")
Map.addLayer(ndvi, vis_ndvi, "MODIS NDVI (masked)")
Map.addLayer(lst_masked, vis_lst, "MODIS LST Day (K, masked)")
Map